In [1]:
import redis
import json
import redis.commands.json as redis_json

REDIS_HOST = 'localhost'
REDIS_PASSWORD = "123456"
REDIS_PORT = 6379
INDEX_NAME = "GameObjectsIdx"
KEY_PREFIX = "GameObjects:"

redis_client = redis.StrictRedis(
    host=REDIS_HOST,
    port=REDIS_PORT,
    password=REDIS_PASSWORD,
    # ssl=USE_SSL,
    decode_responses=True 
)

In [4]:
delete = True
all_data = {}
# Use scan_iter for safe iteration over all keys
for key in redis_client.scan_iter('*'):
    # Determine the type of the key to use the correct retrieval command
    key_type = redis_client.type(key)
    
    try:
        if key_type == 'string':
            # For plain strings (serialized JSON), retrieve the string and parse it
            value = redis_client.get(key)
            all_data[key] = json.loads(value)
        elif key_type == 'ReJSON-RL': # This is the type returned for RedisJSON keys
            # For RedisJSON types, use the JSON.GET command
            value = redis_client.json().get(key)
            all_data[key] = value
        else:
            # Handle other types if necessary (e.g., hash, list, set)
            all_data[key] = f"(Skipped: Non-JSON type '{key_type}')"
    except json.JSONDecodeError:
        # Handle cases where a string key contains non-JSON data
        all_data[key] = f"(Skipped: Invalid JSON in string key)"
    except Exception as e:
        all_data[key] = f"(Error retrieving data: {e})"
        
for key, data in all_data.items():
    print(f"Key: **{key}**")
    if (delete):
        redis_client.delete(key)
    print(f"Value: {data}\n")

In [114]:
#query = "@Tag:{cube} @ComponentsColor:{red} @ComponentsConstantForce:[-inf 9.82]"
query = "@Tag:{cube}"
search_results = redis_client.ft(INDEX_NAME).search(query)
search_results

Result{0 total, docs: []}

In [ ]:
try:
    redis_client.execute_command('FT.DROPINDEX', INDEX_NAME)
    print(f"Index '{INDEX_NAME}' has been deleted successfully.")
except redis.exceptions.ResponseError as e:
    print(f"Index '{INDEX_NAME}' not found. Creating index now...")
    
    create_command_args = [
                'FT.CREATE', INDEX_NAME,
                'ON', 'JSON',
                'PREFIX', 1, KEY_PREFIX,
                'SCHEMA',
                '$.Id', 'AS', 'Id', 'TAG',
                '$.Tag', 'AS', 'Tag', 'TAG',
                '$.Name', 'AS', 'Name', 'TAG',
                '$.Components.ConstantForce.X', 'AS', 'ComponentConstantForceX', 'NUMERIC', 'SORTABLE',
                '$.Components.ConstantForce.Y', 'AS', 'ComponentConstantForceY', 'NUMERIC', 'SORTABLE',
                '$.Components.ConstantForce.Z', 'AS', 'ComponentConstantForceZ', 'NUMERIC', 'SORTABLE',
                '$.Components.Color', 'AS', 'ComponentColor', 'TAG',
                '$.Transform.Position.X', 'AS', 'TransformPositionX', 'NUMERIC', 'SORTABLE',
                '$.Transform.Position.Y', 'AS', 'TransformPositionY', 'NUMERIC', 'SORTABLE',
                '$.Transform.Position.Z', 'AS', 'TransformPositionZ', 'NUMERIC', 'SORTABLE',
                '$.Transform.Rotation.X', 'AS', 'TransformRotationX', 'NUMERIC', 'SORTABLE',
                '$.Transform.Rotation.Y', 'AS', 'TransformRotationY', 'NUMERIC', 'SORTABLE',
                '$.Transform.Rotation.Z', 'AS', 'TransformRotationZ', 'NUMERIC', 'SORTABLE',
                '$.Transform.Scale.X', 'AS', 'TransformScaleX', 'NUMERIC', 'SORTABLE',
                '$.Transform.Scale.Y', 'AS', 'TransformScaleY', 'NUMERIC', 'SORTABLE',
                '$.Transform.Scale.Z', 'AS', 'TransformScaleZ', 'NUMERIC', 'SORTABLE'
            ]
    
    redis_client.execute_command(*create_command_args)

    print(f"Successfully created index '{INDEX_NAME}'.")

Index 'GameObjectsIdx' not found. Creating index now...
Successfully created index 'GameObjectsIdx'.
